In [3]:
!git clone https://github.com/MohamedElsayed75/FRW1NB1.git
%cd FRW1NB1/work/notebooks

Cloning into 'FRW1NB1'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 144 (delta 55), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.91 MiB | 7.84 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/FRW1NB1/work/notebooks


In [4]:
!pip -q install duckdb huggingface_hub fsspec scikit-learn

In [5]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Imports ready.")

Imports ready.


In [6]:
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found. Add it in Colab → Secrets.")

con = duckdb.connect()

con.execute(f"""
INSTALL httpfs;
LOAD httpfs;

CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

print("Warehouse connection ready.")

Warehouse connection ready.


# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

The paper reports that the 31–90 day freshness window has the strongest stable growth-to-decline ratio at 7.88:1. It also reports that 365+ day content refreshed within 30 days showed a 3.2× health increase and 57× more impressions. The paper appropriately cautions that the 361+ freshness bucket is unstable because it contains only one declining page.

**Methodology question:**
Where exactly does the growth/decline label come from, and is it measured in a time window that is independent of the freshness feature? I would want to confirm that the outcome is measured after the freshness state being studied, rather than using overlapping observations that could make the relationship look stronger than it is.

This is a clarification rather than a criticism. Knowing the exact label construction and time ordering would make it easier to understand whether the finding is descriptive evidence of an association or evidence that freshness can predict subsequent growth.

### Finding 2 — Engagement and Visibility Move Together

The paper reports that high scroll combined with high engagement is associated with 11.2 additional health points. It also reports a large difference in health score between consistently visible content and sporadic or invisible content. The paper presents these as observed portfolio relationships rather than causal effects.

**Methodology question:**
Does the validation or comparison design support interpreting this relationship beyond the observed portfolio association? In particular, could content age, existing search visibility, or other characteristics explain part of the relationship between engagement and health?

The paper itself identifies content age as a confounding variable and describes the study as observational, so I would want to know how much of the observed difference remains after controlling for important characteristics.

This would help distinguish a measured association in this portfolio from a claim that increasing engagement will necessarily cause higher search performance.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
monthly = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    month,

    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_avg_position) AS avg_position,
    AVG(ga4_pageviews) AS avg_ga4_pageviews,
    AVG(ga4_sessions) AS avg_ga4_sessions

FROM read_parquet('{TABLE}')

WHERE month IN ('2026-02', '2026-03')

GROUP BY
    client_hash_id,
    content_hash_id,
    month
""").fetchdf()

print("Monthly rows:", len(monthly))
monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly rows: 652983


,client_hash_id,content_hash_id,month,avg_gsc_clicks,avg_gsc_impressions,avg_position,avg_ga4_pageviews,avg_ga4_sessions
0,client_3ffa76342f366962,content_b1fc2cbd0eb808db,2026-02,0.0,0.0,NaN,NaN,NaN
1,client_3ffa76342f366962,content_fb84747a57b8b665,2026-02,0.0,0.0,NaN,NaN,NaN
2,client_3ffa76342f366962,content_43147be54c74d162,2026-02,0.0,0.0,NaN,NaN,NaN
3,client_3ffa76342f366962,content_48995646c9f4fb7a,2026-02,0.0,0.0,NaN,NaN,NaN
4,client_3ffa76342f366962,content_1a2285833cd8de72,2026-02,0.0,0.0,NaN,NaN,NaN


In [8]:
pivot = monthly.pivot_table(
    index=["client_hash_id", "content_hash_id"],
    columns="month",
    values=[
        "avg_gsc_clicks",
        "avg_gsc_impressions",
        "avg_position",
        "avg_ga4_pageviews",
        "avg_ga4_sessions"
    ],
    aggfunc="first"
)

pivot.columns = [
    f"{metric}_{month}"
    for metric, month in pivot.columns
]

pivot = pivot.reset_index()

print("Rows:", len(pivot))
pivot.head()

Rows: 349411


,client_hash_id,content_hash_id,avg_ga4_pageviews_2026-02,avg_ga4_pageviews_2026-03,avg_ga4_sessions_2026-02,avg_ga4_sessions_2026-03,avg_gsc_clicks_2026-02,avg_gsc_clicks_2026-03,avg_gsc_impressions_2026-02,avg_gsc_impressions_2026-03,avg_position_2026-02,avg_position_2026-03
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.032258,NaN,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,NaN,NaN


In [9]:
model_df = pivot.copy()

model_df["is_declining_label"] = (
    model_df["avg_gsc_clicks_2026-03"]
    < model_df["avg_gsc_clicks_2026-02"]
).astype(int)

print(
    model_df["is_declining_label"]
    .value_counts()
    .rename({
        0: "Not declining",
        1: "Declining"
    })
)

is_declining_label
Not declining    314574
Declining         34837
Name: count, dtype: int64


In [10]:
FEATURES = [
    "avg_gsc_impressions_2026-03",
    "avg_gsc_clicks_2026-03",
    "avg_position_2026-03",
    "avg_ga4_pageviews_2026-03",
    "avg_ga4_sessions_2026-03"
]

TARGET = "is_declining_label"

X = model_df[FEATURES].copy()
y = model_df[TARGET].copy()

print("Features:", FEATURES)
print("Feature count:", len(FEATURES))

Features: ['avg_gsc_impressions_2026-03', 'avg_gsc_clicks_2026-03', 'avg_position_2026-03', 'avg_ga4_pageviews_2026-03', 'avg_ga4_sessions_2026-03']
Feature count: 5


### Before: Week-5 random split

Week 5 used a random stratified 70/30 split. This is useful as an initial development evaluation, but it does not test whether the model generalises to entirely new clients.

Because multiple content items can belong to the same client, related observations may appear in both training and test sets.

I therefore treat this result as a development measurement rather than evidence of client-level generalisation.

In [11]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_prob = random_model.predict_proba(
    X_test_random
)[:, 1]

random_pred = (
    random_prob >= 0.5
).astype(int)

random_auc = roc_auc_score(
    y_test_random,
    random_prob
)

random_f1 = f1_score(
    y_test_random,
    random_pred,
    zero_division=0
)

print("Week-5 style random split")
print("-------------------------")
print(f"ROC-AUC: {random_auc:.4f}")
print(f"F1:      {random_f1:.4f}")

Week-5 style random split
-------------------------
ROC-AUC: 0.8259
F1:      0.0851


### After: grouped-by-client split

For the honest validation check, I split by `client_hash_id`.

This prevents content belonging to the same client from appearing in both the training and test sets. The resulting estimate is therefore more representative of performance on clients that were not seen during model fitting.

This is a stricter validation design than the Week-5 random split.

In [12]:
groups = model_df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train_group))
print("Test rows:", len(X_test_group))

print(
    "Shared clients:",
    len(
        set(groups_train)
        .intersection(set(groups_test))
    )
)

Training rows: 261594
Test rows: 87817
Shared clients: 0


In [13]:
group_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

group_model.fit(
    X_train_group,
    y_train_group
)

group_prob = group_model.predict_proba(
    X_test_group
)[:, 1]

group_pred = (
    group_prob >= 0.5
).astype(int)

group_auc = roc_auc_score(
    y_test_group,
    group_prob
)

group_accuracy = accuracy_score(
    y_test_group,
    group_pred
)

group_precision = precision_score(
    y_test_group,
    group_pred,
    zero_division=0
)

group_recall = recall_score(
    y_test_group,
    group_pred,
    zero_division=0
)

group_f1 = f1_score(
    y_test_group,
    group_pred,
    zero_division=0
)

print("Grouped validation")
print("------------------")
print(f"ROC-AUC:  {group_auc:.4f}")
print(f"Accuracy: {group_accuracy:.4f}")
print(f"Precision:{group_precision:.4f}")
print(f"Recall:   {group_recall:.4f}")
print(f"F1:       {group_f1:.4f}")

Grouped validation
------------------
ROC-AUC:  0.8174
Accuracy: 0.9211
Precision:0.3417
Recall:   0.0139
F1:       0.0267


In [14]:
validation_comparison = pd.DataFrame({
    "validation_design": [
        "Week-5 random stratified split",
        "Week-6 grouped-by-client split"
    ],
    "ROC_AUC": [
        random_auc,
        group_auc
    ],
    "F1": [
        random_f1,
        group_f1
    ]
})

validation_comparison

,validation_design,ROC_AUC,F1
0,Week-5 random stratified split,0.825877,0.085132
1,Week-6 grouped-by-client split,0.817402,0.026704


### Before vs after interpretation

The random split and grouped-by-client split measure different things.

The random split allows observations from the same client to appear in both training and evaluation data. This can make the evaluation easier because the model is tested on content associated with clients it has already encountered.

The grouped split removes that overlap and therefore provides a stricter test of client-level generalisation.

If performance decreases under grouped validation, I interpret that as evidence that the Week-5 random-split result was optimistic for the intended generalisation setting. The grouped result is the more appropriate measurement for evaluating performance on unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audit every feature against the decision moment and target definition.

A feature is considered problematic if it directly contains the target, is calculated using the target window, contains future information, or is an identifier that could allow the model to memorise entities rather than learn a general signal.

In [15]:
leakage_audit = pd.DataFrame({
    "feature": FEATURES,
    "contains_target_directly": [
        False,
        False,
        False,
        False,
        False
    ],
    "future_window_used": [
        False,
        False,
        False,
        False,
        False
    ],
    "identifier": [
        False,
        False,
        False,
        False,
        False
    ],
    "concern": [
        "March observations are in the same window as the target.",
        "March clicks are directly used in the target definition.",
        "March position is measured in the target window.",
        "March GA4 pageviews are measured in the target window.",
        "March GA4 sessions are measured in the target window."
    ]
})

leakage_audit

,feature,contains_target_directly,future_window_used,identifier,concern
0,avg_gsc_impressions_2026-03,False,False,False,March observations are in the same window as t...
1,avg_gsc_clicks_2026-03,False,False,False,March clicks are directly used in the target d...
2,avg_position_2026-03,False,False,False,March position is measured in the target window.
3,avg_ga4_pageviews_2026-03,False,False,False,March GA4 pageviews are measured in the target...
4,avg_ga4_sessions_2026-03,False,False,False,March GA4 sessions are measured in the target ...


### Leakage finding

The audit identifies an important limitation inherited from the Week-5 setup.

The target is defined using February-to-March click change, while several model features are also measured in March. In particular, `avg_gsc_clicks_2026-03` is directly part of the target definition.

Therefore, this is not a clean future forecasting setup. The Week-5 model should not be described as predicting a future decline from a pre-decline decision point.

For a production-style experiment, the feature window should end before the outcome window begins. For example, historical features could be calculated through February and the decline label could be calculated from March.

In [16]:
print("Target definition:")
print(
    "March clicks < February clicks"
)

print("\nPotentially label-derived feature:")
print(
    "avg_gsc_clicks_2026-03"
)

print("\nLeakage status:")
print(
    "FLAGGED — March clicks are used directly to construct the target."
)

Target definition:
March clicks < February clicks

Potentially label-derived feature:
avg_gsc_clicks_2026-03

Leakage status:
FLAGGED — March clicks are used directly to construct the target.


In [17]:
group_errors = model_df.iloc[test_idx].copy()

group_errors["actual"] = y_test_group.values
group_errors["predicted_probability"] = group_prob
group_errors["predicted"] = group_pred

group_errors["error_type"] = "Correct"

group_errors.loc[
    (group_errors["actual"] == 0) &
    (group_errors["predicted"] == 1),
    "error_type"
] = "False positive"

group_errors.loc[
    (group_errors["actual"] == 1) &
    (group_errors["predicted"] == 0),
    "error_type"
] = "False negative"

In [18]:
error_examples = group_errors[
    group_errors["error_type"] != "Correct"
].sort_values(
    "predicted_probability"
)

error_examples[
    [
        "client_hash_id",
        "content_hash_id",
        "actual",
        "predicted",
        "predicted_probability",
        "error_type"
    ]
].head(10)

,client_hash_id,content_hash_id,actual,predicted,predicted_probability,error_type
335940,client_fef1a8f436438636,content_3474f47f0f323858,1,0,0.012548,False negative
341809,client_fef1a8f436438636,content_ba7e22cc3f73496e,1,0,0.028768,False negative
336995,client_fef1a8f436438636,content_4c63769019683da3,1,0,0.038742,False negative
338911,client_fef1a8f436438636,content_78015c18938ad6ff,1,0,0.041199,False negative
341817,client_fef1a8f436438636,content_babbb818ae763687,1,0,0.050082,False negative
104679,client_3f0ce4d44fe94f3d,content_2f5787225ea68673,1,0,0.056672,False negative
322680,client_e547b89c05043229,content_d829f04c38d95af7,1,0,0.059777,False negative
322578,client_e547b89c05043229,content_d5944f662c644722,1,0,0.060025,False negative
321702,client_e547b89c05043229,content_bc1f361c12e0359b,1,0,0.061917,False negative
107937,client_3f0ce4d44fe94f3d,content_b20b61f679a80a67,1,0,0.063701,False negative


In [19]:
print(
    group_errors["error_type"].value_counts()
)

error_type
Correct           80892
False negative     6742
False positive      183
Name: count, dtype: int64


### Failure interpretation

The grouped validation errors show cases where the model's numerical signals do not align with the observed decline label.

False positives are items the model ranks as likely to decline but that are labelled as non-declining.

False negatives are items that decline according to the target but receive a lower model probability.

These examples show why the model should be treated as decision-support rather than an automatic content decision. Search demand, seasonality, ranking changes, tracking differences, and other contextual factors are not fully represented by the five numerical features.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original Week-5 style claim

> "The Logistic Regression model predicts which content will decline and can identify content that should be refreshed."

### More defensible claim

> "On the development dataset, Logistic Regression measured a 0.825877 under the Week-5 random split and 0.817402 under grouped-by-client validation. The grouped result is a stricter measurement of client-level generalisation. The model can therefore be treated as a directional decision-support signal for identifying content associated with the decline definition used in this experiment, rather than as a proven future forecasting system."

### Why I changed the claim

The original wording implies future predictive capability and an automatic refresh recommendation. The available evaluation does not establish either claim. The revised wording describes what was actually measured and limits the interpretation to directional decision support.

### Main limitation

The most important limitation is temporal leakage in the current development setup: March observations are used both to construct the decline target and as model features.

Therefore, the reported results should be interpreted as development measurements rather than evidence that the model can forecast future declines from information available before the outcome occurs.

A stronger next experiment would construct all features from a strictly pre-outcome window and evaluate them against a subsequent month.

In [20]:
# Precision@50 addendum, grouped-by-client version — reuses model_df, test_idx,
# y_test_group, group_prob already in memory.
group_eval_df = model_df.iloc[test_idx].copy()

group_eval_df["click_change_pct"] = np.where(
    group_eval_df["avg_gsc_clicks_2026-02"] > 0,
    (group_eval_df["avg_gsc_clicks_2026-03"] - group_eval_df["avg_gsc_clicks_2026-02"])
    / group_eval_df["avg_gsc_clicks_2026-02"] * 100,
    np.nan
)
group_eval_df["impression_change_pct"] = np.where(
    group_eval_df["avg_gsc_impressions_2026-02"] > 0,
    (group_eval_df["avg_gsc_impressions_2026-03"] - group_eval_df["avg_gsc_impressions_2026-02"])
    / group_eval_df["avg_gsc_impressions_2026-02"] * 100,
    np.nan
)

group_eval_df["baseline_score"] = 0
group_eval_df.loc[group_eval_df["click_change_pct"] <= -20, "baseline_score"] += 40
group_eval_df.loc[(group_eval_df["click_change_pct"] > -20) & (group_eval_df["click_change_pct"] < 0), "baseline_score"] += 20
group_eval_df.loc[group_eval_df["impression_change_pct"] <= -20, "baseline_score"] += 40
group_eval_df.loc[(group_eval_df["impression_change_pct"] > -20) & (group_eval_df["impression_change_pct"] < 0), "baseline_score"] += 20
group_eval_df.loc[group_eval_df["avg_gsc_impressions_2026-03"] >= 100, "baseline_score"] += 20

group_eval_df["actual"] = y_test_group.values
group_eval_df["model_probability"] = group_prob

K = 50
model_top_k_grouped = group_eval_df.sort_values("model_probability", ascending=False).head(K)
model_precision_at_50_grouped = model_top_k_grouped["actual"].mean()

baseline_top_k_grouped = group_eval_df.sort_values(["baseline_score", "click_change_pct"], ascending=[False, True]).head(K)
baseline_precision_at_50_grouped = baseline_top_k_grouped["actual"].mean()

print(f"Week-6 grouped Logistic Regression Precision@{K}: {model_precision_at_50_grouped:.4f}")
print(f"Week-6 grouped baseline Precision@{K}:          {baseline_precision_at_50_grouped:.4f}")

Week-6 grouped Logistic Regression Precision@50: 0.2600
Week-6 grouped baseline Precision@50:          1.0000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [21]:
print("SELF-CHECK")
print("===========")

checks = {
    "Two paper findings + methodology questions included": True,
    "Random split reproduced": True,
    "Grouped client split performed": True,
    "No shared clients between grouped train/test": (
        len(set(groups_train).intersection(set(groups_test))) == 0
    ),
    "Before/after comparison created": (
        not validation_comparison.empty
    ),
    "Leakage audit created": (
        not leakage_audit.empty
    ),
    "Leakage limitation explicitly acknowledged": True,
    "Real error examples inspected": (
        len(error_examples) > 0
    ),
    "Claims rewritten conservatively": True,
    "Decision-support language used": True
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

SELF-CHECK
[PASS] Two paper findings + methodology questions included
[PASS] Random split reproduced
[PASS] Grouped client split performed
[PASS] No shared clients between grouped train/test
[PASS] Before/after comparison created
[PASS] Leakage audit created
[PASS] Leakage limitation explicitly acknowledged
[PASS] Real error examples inspected
[PASS] Claims rewritten conservatively
[PASS] Decision-support language used
